<img src="https://raw.githubusercontent.com/Databricks-BR/lab_agosto_2025/main/images/head_lab.png">

## LAB 03 — Entregável CEO (challenge 001)

Diagnóstico para o CEO: causa raiz cruzando as 5 tabelas, segmentos em risco com `account_id`, ações priorizadas com ARR.

1. Rode o **LAB 01** antes.
2. Widgets iguais aos outros labs.
3. Cluster ou serverless → **Run All**.
4. Em cada `display`, use o ícone de gráfico do Databricks para fixar o tile no AI/BI.

Label canônico = `ravenstack_churn_events`. Não use `accounts.churn_flag` como verdade.

In [ ]:
def widget_or_default(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "ravenstack")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")
print(f"{catalog_name}.{schema_name}")

### Tile 1 — Volume: contas vs evento de churn

Cruzamento `accounts` × `churn_events`. O flag da conta **não** entra no KPI.

In [ ]:
display(
    spark.sql(
        """
SELECT
  (SELECT COUNT(*) FROM ravenstack_accounts) AS accounts,
  (SELECT COUNT(DISTINCT account_id) FROM ravenstack_churn_events) AS accounts_with_event,
  (SELECT COUNT(*) FROM ravenstack_churn_events) AS churn_events,
  (SELECT ROUND(SUM(refund_amount_usd), 2) FROM ravenstack_churn_events) AS refund_usd
"""
    )
)

### Tile 2 — Segmentos em risco (industry × plan × referral)

Taxa = contas com pelo menos um `churn_event_id`.

In [ ]:
display(
    spark.sql(
        """
SELECT
  a.industry,
  a.plan_tier,
  a.referral_source,
  COUNT(*) AS accounts,
  SUM(CASE WHEN e.account_id IS NOT NULL THEN 1 ELSE 0 END) AS with_event,
  ROUND(AVG(CASE WHEN e.account_id IS NOT NULL THEN 1.0 ELSE 0.0 END), 3) AS event_rate
FROM ravenstack_accounts a
LEFT JOIN (SELECT DISTINCT account_id FROM ravenstack_churn_events) e
  ON a.account_id = e.account_id
GROUP BY a.industry, a.plan_tier, a.referral_source
HAVING COUNT(*) >= 5
ORDER BY event_rate DESC, accounts DESC
"""
    )
)

### Tile 3 — Contas nomeadas (ARR vigente + motivo + tickets)

Assinatura vigente = `end_date` nulo. Lista o que o CEO consegue acionar nesta semana.

In [ ]:
display(
    spark.sql(
        """
SELECT
  a.account_id,
  a.account_name,
  a.industry,
  a.plan_tier,
  a.country,
  ROUND(s.arr_amount, 2) AS arr_vigente,
  e.reason_code,
  ROUND(e.refund_amount_usd, 2) AS refund_usd,
  e.preceding_downgrade_flag,
  t.tickets,
  ROUND(t.avg_csat, 2) AS avg_csat
FROM ravenstack_accounts a
INNER JOIN (
  SELECT account_id, reason_code, refund_amount_usd, preceding_downgrade_flag,
         ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY churn_date DESC) AS rn
  FROM ravenstack_churn_events
) e ON a.account_id = e.account_id AND e.rn = 1
LEFT JOIN (
  SELECT account_id, arr_amount,
         ROW_NUMBER() OVER (PARTITION BY account_id ORDER BY start_date DESC) AS rn
  FROM ravenstack_subscriptions
  WHERE end_date IS NULL
) s ON a.account_id = s.account_id AND s.rn = 1
LEFT JOIN (
  SELECT account_id, COUNT(*) AS tickets, AVG(satisfaction_score) AS avg_csat
  FROM ravenstack_support_tickets
  GROUP BY account_id
) t ON a.account_id = t.account_id
ORDER BY s.arr_amount DESC NULLS LAST
LIMIT 25
"""
    )
)

### Tile 4 — Causa raiz (downgrade + usage + tickets, não o flag)

Testa a história do CEO: uso cresceu? CS diz satisfação ok?

In [ ]:
display(
    spark.sql(
        """
SELECT
  CASE WHEN e.account_id IS NOT NULL THEN 'has_churn_event' ELSE 'no_event' END AS cohort,
  COUNT(DISTINCT a.account_id) AS accounts,
  ROUND(AVG(u.avg_usage), 2) AS avg_usage_count,
  ROUND(AVG(u.avg_errors), 2) AS avg_error_count,
  ROUND(AVG(t.avg_csat), 2) AS avg_csat,
  ROUND(AVG(t.avg_hours), 2) AS avg_resolution_hours,
  SUM(CASE WHEN d.account_id IS NOT NULL THEN 1 ELSE 0 END) AS accounts_with_downgrade
FROM ravenstack_accounts a
LEFT JOIN (SELECT DISTINCT account_id FROM ravenstack_churn_events) e
  ON a.account_id = e.account_id
LEFT JOIN (
  SELECT s.account_id, AVG(f.usage_count) AS avg_usage, AVG(f.error_count) AS avg_errors
  FROM ravenstack_subscriptions s
  JOIN ravenstack_feature_usage f ON s.subscription_id = f.subscription_id
  GROUP BY s.account_id
) u ON a.account_id = u.account_id
LEFT JOIN (
  SELECT account_id, AVG(satisfaction_score) AS avg_csat, AVG(resolution_time_hours) AS avg_hours
  FROM ravenstack_support_tickets
  GROUP BY account_id
) t ON a.account_id = t.account_id
LEFT JOIN (
  SELECT DISTINCT account_id FROM ravenstack_subscriptions WHERE downgrade_flag = true
) d ON a.account_id = d.account_id
GROUP BY CASE WHEN e.account_id IS NOT NULL THEN 'has_churn_event' ELSE 'no_event' END
"""
    )
)

### Tile 5 — Ranking por país (volume de eventos, não a taxa)

Prioridade geográfica: quem concentra `churn_events`. A taxa fica ao lado, não ordena.

In [ ]:
display(
    spark.sql(
        """
SELECT
  a.country,
  COUNT(DISTINCT a.account_id) AS accounts,
  COUNT(DISTINCT e.account_id) AS accounts_with_event,
  ROUND(
    COUNT(DISTINCT e.account_id) / NULLIF(COUNT(DISTINCT a.account_id), 0),
    3
  ) AS event_rate
FROM ravenstack_accounts a
LEFT JOIN ravenstack_churn_events e ON a.account_id = e.account_id
GROUP BY a.country
ORDER BY accounts_with_event DESC
"""
    )
)

### Ações para o CEO (prioridade)

| # | Dono | Ação | Por quê | Impacto |
| --- | --- | --- | --- | --- |
| 1 | CS | Lista das 25 contas do tile 3 — playbook nesta semana | ARR vigente + `reason_code` visíveis | Recuperar refund e segurar ARR |
| 2 | Produto | Investigar `reason_code` mais frequente com `preceding_downgrade_flag` | Cruzamento evento × assinatura, não o flag | Reduzir churn no segmento do tile 2 |
| 3 | CS | Tickets com `resolution_time_hours` alto nas contas com evento | Tile 4 testa “satisfação ok” | Horas e CSAT no mesmo recorte |

Lab 04 (`04_ANALISE_retention`) aprofunda o EDA. Este notebook é o relatório do CEO.

### Painéis gráficos

Ranking, galeria, classificação, segmentação, segmentação de imagem, estatísticas do modelo, matriz de confusão, curva de aprendizado e viés/variância. Calculados só com colunas reais do dump.


In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / '03_LAB_Dashboard', HERE.parent / 'lib', HERE.parent.parent / 'lib'):
    if (candidate / 'lab03_panels' / '__init__.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('lab03_panels não encontrado')

from lab03_panels import build_churn_panels
from lab03_panels.render import render_panels

try:
    show = display
except NameError:
    show = print

panels = build_churn_panels()
render_panels(panels, show)
print('secoes', list(panels.panels))
